# Exercise 12 - Custom implementation for LayerNormalization

In [8]:
import tensorflow as tf
import numpy as np
from tensorflow import keras

In [6]:
class LayerNormalization(keras.layers.Layer):
    def __init__(self, epsilon=0.001, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.alpha = self.add_weight(name="alpha", shape=input_shape[-1:], dtype=tf.float32, initializer="ones", trainable=True)
        self.beta = self.add_weight(name="beta", shape=input_shape[-1:], dtype=tf.float32, initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        mean, variance = tf.nn.moments(inputs, axes=-1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(variance + self.epsilon)
        return self.alpha * normalized + self.beta

In [7]:
tf.random.set_seed(42)
X = tf.random.normal(shape=(2, 3, 4))

custom_layer = LayerNormalization()
keras_layer = keras.layers.LayerNormalization(epsilon=0.001, axis=-1)

custom_output = custom_layer(X)
keras_output = keras_layer(X)

print("Custom output:\n", custom_output.numpy())
print("\nKeras output:\n", keras_output.numpy())

print("\nMaximum absolute difference:", tf.reduce_max(tf.abs(custom_output - keras_output)).numpy())
tf.debugging.assert_near(custom_output, keras_output, atol=1e-5)

Custom output:
 [[[ 0.96849823 -0.58752805  0.9578133  -1.3387837 ]
  [-1.4521675  -0.16931194  0.28904676  1.3324326 ]
  [ 1.4287091   0.20376745 -1.3555722  -0.27690446]]

 [[-0.02667039 -1.3738577  -0.05036796  1.4508961 ]
  [ 0.10058279 -1.6465416   0.9754772   0.57048154]
  [ 1.2099249   0.31996492 -1.5562718   0.02638202]]]

Keras output:
 [[[ 0.96849823 -0.5875281   0.95781326 -1.3387836 ]
  [-1.4521673  -0.16931194  0.28904676  1.3324325 ]
  [ 1.4287093   0.20376745 -1.3555722  -0.27690446]]

 [[-0.0266704  -1.3738577  -0.05036795  1.450896  ]
  [ 0.10058281 -1.6465416   0.9754772   0.5704815 ]
  [ 1.2099249   0.319965   -1.5562719   0.02638209]]]

Maximum absolute difference: 2.3841858e-07


# Exercise 13 - Custom training loop for Fashion MNIST dataset

In [9]:
keras.utils.set_random_seed(42)
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

X_train_full = X_train_full.astype("float32") / 255.
X_test = X_test.astype("float32") / 255.

X_train, y_train = X_train_full[5000:], y_train_full[5000:]
X_val, y_val = X_train_full[:5000], y_train_full[:5000]

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10_000).batch(32)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(32)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 4us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step 
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [10]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

In [11]:
loss_fn = keras.losses.SparseCategoricalCrossentropy()

lower_optimizer = keras.optimizers.SGD(learning_rate=1e-3)
upper_optimizer = keras.optimizers.Adam(learning_rate=1e-3)

train_loss_metric = keras.metrics.Mean()
train_accuracy_metric = keras.metrics.SparseCategoricalAccuracy()

val_loss_metric = keras.metrics.Mean()
val_accuracy_metric = keras.metrics.SparseCategoricalAccuracy()

In [12]:
epochs = 10
lower_vars = (model.layers[1].trainable_variables)
upper_vars = (model.layers[2].trainable_variables + model.layers[3].trainable_variables)

for epoch in range(1, epochs + 1):
    train_loss_metric.reset_state()
    train_accuracy_metric.reset_state()
    val_loss_metric.reset_state()
    val_accuracy_metric.reset_state()
    print(f"\nEpoch {epoch}/{epochs}")

    for iteration, (X_batch, y_batch) in enumerate(train_dataset, start=1):
        with tf.GradientTape() as tape:
            y_pred = model(X_batch, training=True)
            loss = loss_fn(y_batch, y_pred)

        grads = tape.gradient(loss, lower_vars + upper_vars)
        lower_grads = grads[:len(lower_vars)]
        upper_grads = grads[len(lower_vars):]

        lower_optimizer.apply_gradients(zip(lower_grads, lower_vars))
        upper_optimizer.apply_gradients(zip(upper_grads, upper_vars))

        train_loss_metric.update_state(loss)
        train_accuracy_metric.update_state(y_batch, y_pred)

        print(f"\rEpoch {epoch}/{epochs} - "
            f"Iteration {iteration + 1}/{len(X_train) // 32} - "
            f"Loss: {train_loss_metric.result():.4f} - "
            f"Accuracy: {train_accuracy_metric.result():.4f}",
            end="")

    for X_batch, y_batch in val_dataset:
        val_pred = model(X_batch, training=False)
        val_loss = loss_fn(y_batch, val_pred)

        val_loss_metric.update_state(val_loss)
        val_accuracy_metric.update_state(y_batch, val_pred)

    print(f"\nValidation loss: {val_loss_metric.result():.4f} - "
        f"Validation accuracy: {val_accuracy_metric.result():.4f}")


Epoch 1/10
Epoch 1/10 - Iteration 1720/1718 - Loss: 0.5597 - Accuracy: 0.8068
Validation loss: 0.4523 - Validation accuracy: 0.8348

Epoch 2/10
Epoch 2/10 - Iteration 1720/1718 - Loss: 0.4163 - Accuracy: 0.8513
Validation loss: 0.3925 - Validation accuracy: 0.8602

Epoch 3/10
Epoch 3/10 - Iteration 1720/1718 - Loss: 0.3759 - Accuracy: 0.8645
Validation loss: 0.3692 - Validation accuracy: 0.8714

Epoch 4/10
Epoch 4/10 - Iteration 1720/1718 - Loss: 0.3467 - Accuracy: 0.8723
Validation loss: 0.3518 - Validation accuracy: 0.8698

Epoch 5/10
Epoch 5/10 - Iteration 1720/1718 - Loss: 0.3239 - Accuracy: 0.8817
Validation loss: 0.3411 - Validation accuracy: 0.8764

Epoch 6/10
Epoch 6/10 - Iteration 1720/1718 - Loss: 0.3053 - Accuracy: 0.8873
Validation loss: 0.3339 - Validation accuracy: 0.8778

Epoch 7/10
Epoch 7/10 - Iteration 1720/1718 - Loss: 0.2896 - Accuracy: 0.8940
Validation loss: 0.3303 - Validation accuracy: 0.8794

Epoch 8/10
Epoch 8/10 - Iteration 1720/1718 - Loss: 0.2753 - Accurac

In [14]:
model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

Test loss: 0.3543
Test accuracy: 0.8777
